In [19]:
%load_ext autoreload
%autoreload 2

In [117]:
import pandas as pd
from utils import download_s3_prefixes, format_rtc_input_data, get_rtc_input_data, validate_inputs_of_one
from pathlib import Path
import rasterio
from dist_s1_enumerator import enumerate_one_dist_s1_product

# Download

In [27]:
df_sds_prod = pd.read_csv('DIST-S1 Time Series Test Results 2025-12-04 - Test Results 2025-12-04.csv')
df_sds_prod.dropna(inplace=True)
df_sds_prod.head()

,MGRS Tile ID,PCM Run Params,Granule ID,S3 Location,URL
0,11SLT,"--product-id-time: 11SLT_2,20250502T015849Z",OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,https://opera-int-rs-fwd.s3.us-west-2.amazonaw...
1,11SLT,"--product-id-time: 11SLT_3,20250502T140102Z",OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,https://opera-int-rs-fwd.s3.us-west-2.amazonaw...
2,11SLT,"--product-id-time: 11SLT_0,20250503T014926Z",OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,https://opera-int-rs-fwd.s3.us-west-2.amazonaw...
3,11SLT,"--product-id-time: 11SLT_1,20250503T135141Z",OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T135131Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,https://opera-int-rs-fwd.s3.us-west-2.amazonaw...
4,11SLT,"--product-id-time: 11SLT_2,20250508T015734Z",OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T015723Z...,s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L...,https://opera-int-rs-fwd.s3.us-west-2.amazonaw...


In [28]:
urls_11SLT = sorted(df_sds_prod[df_sds_prod['MGRS Tile ID'] == '11SLT']['S3 Location'].tolist())
urls_11SLT

['s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z_20251204T032024Z_S1C_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T135131Z_20251204T042653Z_S1C_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T015723Z_20251204T052634Z_S1C_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T135942Z_20251204T061957Z_S1C_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T015029Z_20251204T071931Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T135246Z_20251204T082126Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1

In [29]:
urls_30UXC = (df_sds_prod[df_sds_prod['MGRS Tile ID'] == '30UXC']['S3 Location'].tolist())
urls_30UXC

['s3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250510T061513Z_20251204T042210Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250513T174956Z_20251204T053238Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250518T175807Z_20251204T061812Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250521T062210Z_20251204T072543Z_S1C_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250522T061512Z_20251204T082205Z_S1A_30_v0.1/',
 's3://opera-int-rs-fwd/products/DIST_S1

In [32]:
paths_30SLT = download_s3_prefixes(urls_11SLT, out_dir='prods/11SLT')

opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z_20251204T032024Z_S1C_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T135131Z_20251204T042653Z_S1C_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T015723Z_20251204T052634Z_S1C_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T135942Z_20251204T061957Z_S1C_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T015029Z_20251204T071931Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T135246Z_20251204T082126Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250514T015837Z_20251204T093038Z_S1A_30_v0.1/
o

Downloading: 100%|████████████| 540/540 [03:35<00:00,  2.51it/s]


# Check Confirmation of prior product

In [69]:
status_paths_11SLT = sorted(list(Path('prods/11SLT/').rglob('*STATUS.tif')), key=lambda p: p.name)
status_paths_11SLT[:3]

[PosixPath('prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z_20251204T032024Z_S1C_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z_20251204T032024Z_S1C_30_v0.1_GEN-DIST-STATUS.tif')]

In [31]:
paths_30UXC = download_s3_prefixes(urls_30UXC, out_dir='prods/30UXC')

opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250510T061513Z_20251204T042210Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250513T174956Z_20251204T053238Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250518T175807Z_20251204T061812Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250521T062210Z_20251204T072543Z_S1C_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250522T061512Z_20251204T082205Z_S1A_30_v0.1/
opera-int-rs-fwd/products/DIST_S1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250524T175655Z_20251204T092120Z_S1C_30_v0.1/
o

Downloading: 100%|████████████| 510/510 [05:15<00:00,  1.61it/s]


In [50]:
status_paths_30UXC = sorted(list(Path('prods/30UXC/').rglob('*STATUS.tif')), key=lambda p: p.name)
status_paths_30UXC[:3]

[PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1_GEN-DIST-STATUS.tif')]

In [65]:
def get_prior_prod(one_tif_path: str) -> str:
    with rasterio.open(one_tif_path) as ds:
        tags = ds.tags()
    prior_prod = Path(tags['prior_dist_s1_product']).name
    return prior_prod

def verify_time_series(paths: list[str]) -> list[bool]:
    results = []
    for k, path in enumerate(paths):
        prior_prod = get_prior_prod(path)
        print(f'{prior_prod}')
        if k == 0:
            success = prior_prod == 'None'
            results.append((path.name, success))
        else:
            expected_prior_prod = paths[k-1].name.replace('_GEN-DIST-STATUS.tif', '')
            success = prior_prod == expected_prior_prod
            results.append((path.name, success))
    return results

In [67]:
verify_time_series(status_paths_30UXC)

None
OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250510T061513Z_20251204T042210Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250513T174956Z_20251204T053238Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250518T175807Z_20251204T061812Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250521T062210Z_20251204T072543Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250522T061512Z_20251204T082205Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250524T175655Z_20251204T092120Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250527T062323Z_20251204T111714Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250528T061401Z_20251204T121957Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250530T175806Z_20251204T131316Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T30UXC_20250531T174845Z_20251204T141719Z_S1C_30_v0.1
OPERA_L

[('OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250510T061513Z_20251204T042210Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250513T174956Z_20251204T053238Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250518T175807Z_20251204T061812Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250521T062210Z_20251204T072543Z_S1C_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250522T061512Z_20251204T082205Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T30UXC_20250524T175655Z_20251204T092120Z_S1C_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L

In [70]:
verify_time_series(status_paths_11SLT)

None
OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z_20251204T032024Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T135131Z_20251204T042653Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T015723Z_20251204T052634Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T135942Z_20251204T061957Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T015029Z_20251204T071931Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T135246Z_20251204T082126Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250514T015837Z_20251204T093038Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250514T140056Z_20251204T102000Z_S1A_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250515T014916Z_20251204T111909Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250515T135132Z_20251204T122045Z_S1C_30_v0.1
OPERA_L3_DIST-ALERT-S1_T11SLT_20250520T015724Z_20251204T131558Z_S1C_30_v0.1
OPERA_L

[('OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T014915Z_20251204T032024Z_S1C_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250503T135131Z_20251204T042653Z_S1C_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T015723Z_20251204T052634Z_S1C_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250508T135942Z_20251204T061957Z_S1C_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T015029Z_20251204T071931Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250509T135246Z_20251204T082126Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L3_DIST-ALERT-S1_T11SLT_20250514T015837Z_20251204T093038Z_S1A_30_v0.1_GEN-DIST-STATUS.tif',
  True),
 ('OPERA_L

# Validate Inputs


In [119]:
def validate_inputs_wrapper(path):
    try:
        validate_inputs_of_one(path)
    except ValueError as e:
        print('Issue with product: ', path)
        print(e)
    return

In [120]:
[validate_inputs_wrapper(p) for p in status_paths_11SLT]

Searching for post-images for track 137 in MGRS tile 11SLT
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


Windows: 100%|█| 3/3 [00:17<00:00,  5.73s/i


Issue with product:  prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T015838Z_20251204T012405Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
Burst ID mismatch - expected: ['T137-292311-IW3', 'T137-292312-IW2', 'T137-292312-IW3', 'T137-292313-IW2', 'T137-292313-IW3', 'T137-292314-IW3', 'T137-292315-IW3', 'T137-292316-IW2', 'T137-292316-IW3', 'T137-292317-IW2', 'T137-292317-IW3', 'T137-292318-IW2', 'T137-292318-IW3'], found: ['T137-292311-IW3', 'T137-292312-IW2', 'T137-292312-IW3', 'T137-292313-IW2', 'T137-292313-IW3', 'T137-292314-IW2', 'T137-292314-IW3', 'T137-292315-IW2', 'T137-292315-IW3', 'T137-292316-IW2', 'T137-292316-IW3', 'T137-292317-IW2', 'T137-292317-IW3', 'T137-292318-IW2', 'T137-292318-IW3']
Pre RTC IDs found but not expected: ['OPERA_L2_RTC-S1_T137-292314-IW2_20240320T015851Z', 'OPERA_L2_RTC-S1_T137-292314-IW2_20240401T015852Z', 'OPERA_L2_RTC-S1_T137-292314-IW2_20240413T015851Z', 'OPERA_L2_RTC-S1_T137-292314-I

Windows: 100%|█| 3/3 [00:09<00:00,  3.10s/i


Issue with product:  prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250502T140056Z_20251204T021103Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
Burst ID mismatch - expected: ['T144-308024-IW1', 'T144-308025-IW1', 'T144-308026-IW1', 'T144-308027-IW1', 'T144-308029-IW1', 'T144-308030-IW1', 'T144-308031-IW1'], found: ['T144-308024-IW1', 'T144-308025-IW1', 'T144-308026-IW1', 'T144-308027-IW1', 'T144-308028-IW1', 'T144-308029-IW1', 'T144-308030-IW1', 'T144-308031-IW1']
Pre RTC IDs found but not expected: ['OPERA_L2_RTC-S1_T144-308028-IW1_20240320T140114Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20240401T140115Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20240413T140113Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20240425T140115Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20220331T140103Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20220412T140103Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20220424T140104Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20230407T140109Z', 'OPERA_L2_RTC-S1_

...

Issue with product:  prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250526T015837Z_20251204T162050Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250526T015837Z_20251204T162050Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
Burst ID mismatch - expected: ['T137-292311-IW3', 'T137-292312-IW2', 'T137-292312-IW3', 'T137-292313-IW2', 'T137-292313-IW3', 'T137-292314-IW3', 'T137-292315-IW3', 'T137-292316-IW2', 'T137-292316-IW3', 'T137-292317-IW2', 'T137-292317-IW3', 'T137-292318-IW2', 'T137-292318-IW3'], found: ['T137-292311-IW3', 'T137-292312-IW2', 'T137-292312-IW3', 'T137-292313-IW2', 'T137-292313-IW3', 'T137-292314-IW2', 'T137-292314-IW3', 'T137-292315-IW2', 'T137-292315-IW3', 'T137-292316-IW2', 'T137-292316-IW3', 'T137-292317-IW2', 'T137-292317-IW3', 'T137-292318-IW2', 'T137-292318-IW3']
Pre RTC IDs found but not expected: ['OPERA_L2_RTC-S1_T137-292314-IW2_20240413T015851Z', 'OPERA_L2_RTC-S1_T137-292314-IW2_20240425T015852Z', 'OPERA_L2_RTC-S1_T137-292314-IW2_20240507T015853Z', 'OPERA_L2_RTC-S1_T137-292314-I

Windows: 100%|█| 3/3 [00:08<00:00,  2.92s/i


Issue with product:  prods/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250526T140056Z_20251204T171313Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250526T140056Z_20251204T171313Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
Burst ID mismatch - expected: ['T144-308024-IW1', 'T144-308025-IW1', 'T144-308026-IW1', 'T144-308027-IW1', 'T144-308029-IW1', 'T144-308030-IW1', 'T144-308031-IW1'], found: ['T144-308024-IW1', 'T144-308025-IW1', 'T144-308026-IW1', 'T144-308027-IW1', 'T144-308028-IW1', 'T144-308029-IW1', 'T144-308030-IW1', 'T144-308031-IW1']
Pre RTC IDs found but not expected: ['OPERA_L2_RTC-S1_T144-308028-IW1_20240413T140113Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20240425T140115Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20240507T140115Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20240519T140115Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20220424T140104Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20220506T140104Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20220518T140105Z', 'OPERA_L2_RTC-S1_T144-308028-IW1_20230501T140110Z', 'OPERA_L2_RTC-S1_

Windows: 100%|█| 3/3 [00:10<00:00,  3.36s/i


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [122]:
[validate_inputs_wrapper(p) for p in status_paths_30UXC]

Searching for post-images for track 132 in MGRS tile 30UXC
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


Windows: 100%|█████████████████████████████████████████████████████| 3/3 [00:17<00:00,  5.87s/it]


Issue with product:  prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
Pre RTC IDs expected but not found: ['OPERA_L2_RTC-S1_T132-281681-IW2_20240307T175002Z', 'OPERA_L2_RTC-S1_T132-281682-IW2_20240307T175005Z', 'OPERA_L2_RTC-S1_T132-281683-IW1_20240307T175007Z', 'OPERA_L2_RTC-S1_T132-281683-IW2_20240307T175008Z']
Searching for post-images for track 154 in MGRS tile 30UXC
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


Windows: 100%|█████████████████████████████████████████████████████| 3/3 [00:09<00:00,  3.16s/it]


Searching for post-images for track 30 in MGRS tile 30UXC
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


...

Searching for post-images for track 30 in MGRS tile 30UXC
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


Windows: 100%|█████████████████████████████████████████████████████| 3/3 [00:09<00:00,  3.08s/it]


Searching for post-images for track 132 in MGRS tile 30UXC
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


Windows: 100%|█████████████████████████████████████████████████████| 3/3 [00:18<00:00,  6.23s/it]


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

# Debugging/Hist

In [116]:
LAYER_TIF = status_paths_30UXC[3]
status_paths_30UXC

[PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250501T174957Z_20251204T014648Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250503T062323Z_20251204T021348Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250506T175807Z_20251204T032507Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250510T061513Z_20251204T042210Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250510T061513Z_20251204T042210Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('prods/30UXC/OPERA_L3_DIST-ALERT-S1_T30UXC_20250513T174956Z_20251204T053238Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T30UXC_20250513T174956Z_20251204T053238Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 Posi

In [108]:
data = get_rtc_input_data(LAYER_TIF)

df_prod = format_rtc_input_data(data)

track_numbers = df_prod.track_number.unique().tolist()
if len(track_numbers) > 1:
    if abs(track_numbers[0] - track_numbers[1]) > 1:
        raise ValueError(f'too many track numbers present: {track_numbers}')
post_ind = df_prod.input_category == 'post'
df_post = df_prod[post_ind].reset_index(drop=True)

pre_ind = df_prod.input_category == 'pre'
df_pre = df_prod[pre_ind].reset_index(drop=True)

In [109]:
from typing import Dict, Tuple, Any

def count_acquisitions_by_window(
    df_prod: pd.DataFrame, ref_date: Any
) -> Dict[str, Tuple[int, int, int]]:

    df_prod['days_ago'] = (ref_date - df_prod['acq_dt']).dt.days

    # 3. Define the bins for 'days_ago'
    # Bins: (365, 730], (730, 1095], (1095, 1460]
    bins = [365, 730, 1095, 1500]
    labels = ['W1', 'W2', 'W3'] 

    # Use pd.cut to assign each row to a time window
    df_prod['time_window'] = pd.cut(df_prod['days_ago'], bins=bins, labels=labels, right=False)

    # 4. Group by 'burst_id' and 'time_window', then unstack to get the counts
    counts_df = (
        df_prod.dropna(subset=['time_window'])
        .groupby(['jpl_burst_id', 'time_window'])
        .size()
        .unstack(fill_value=0)
    )

    # 5. Ensure all 3 window columns exist and are in the correct order
    for label in labels:
        if label not in counts_df.columns:
            counts_df[label] = 0

    counts_df = counts_df[labels]

    # 6. Convert the resulting DataFrame to the desired dictionary format
    result_dict = counts_df.apply(tuple, axis=1).to_dict()

    return result_dict


def get_ordered_dates_by_burst(df: pd.DataFrame) -> dict:
    df_copy = df.copy()
    def sort_and_format_dates(series: pd.Series) -> list[str]:
        sorted_dates = series.sort_values(ascending=False)
        return sorted_dates.dt.strftime('%Y-%m-%d').tolist()

    ordered_dates = (
        df_copy.groupby('jpl_burst_id')['acq_dt']
        .apply(sort_and_format_dates)
        .to_dict()
    )

    return ordered_dates

In [112]:
count_acquisitions_by_window(df_pre, df_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_85556/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T081-172597-IW2': (4, 3, 3),
 'T081-172597-IW3': (4, 3, 3),
 'T081-172598-IW2': (4, 3, 3),
 'T081-172598-IW3': (3, 3, 3),
 'T081-172599-IW2': (4, 3, 3),
 'T081-172599-IW3': (4, 3, 3),
 'T081-172600-IW2': (3, 3, 3),
 'T081-172600-IW3': (3, 3, 3),
 'T081-172601-IW2': (4, 3, 3),
 'T081-172601-IW3': (4, 3, 3),
 'T081-172602-IW2': (4, 3, 3),
 'T081-172602-IW3': (4, 3, 3),
 'T081-172603-IW2': (4, 3, 3),
 'T081-172603-IW3': (4, 3, 3)}

In [114]:
df_product_expected = enumerate_one_dist_s1_product(
    df_prod.mgrs_tile_id.iloc[0],
    track_number=track_numbers[0],
    post_date=str(df_post.acq_dt.min().date()),
    lookback_strategy='multi_window',
    delta_lookback_days=(1095, 730, 365),
    max_pre_imgs_per_burst=MAX_PER_BURST
)
df_product_expected['opera_id_trunc'] = df_product_expected.opera_id.map(lambda opera_id: '_'.join(opera_id.split('_')[:5]))

post_ind = df_product_expected.input_category == 'post'
df_product_expected_post = df_product_expected[post_ind].reset_index(drop=True)

pre_ind = df_product_expected.input_category == 'pre'
df_product_expected_pre = df_product_expected[pre_ind].reset_index(drop=True)

Searching for post-images for track 81 in MGRS tile 30UXC
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (3, 3, 4)


Windows: 100%|████████████████████| 3/3 [00:14<00:00,  4.99s/it]


In [115]:
count_acquisitions_by_window(df_product_expected_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_85556/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T081-172597-IW2': (4, 3, 3),
 'T081-172597-IW3': (4, 3, 3),
 'T081-172598-IW2': (4, 3, 3),
 'T081-172598-IW3': (4, 3, 3),
 'T081-172599-IW2': (4, 3, 3),
 'T081-172599-IW3': (4, 3, 3),
 'T081-172600-IW2': (4, 3, 3),
 'T081-172600-IW3': (4, 3, 3),
 'T081-172601-IW2': (4, 3, 3),
 'T081-172601-IW3': (4, 3, 3),
 'T081-172602-IW2': (4, 3, 3),
 'T081-172602-IW3': (4, 3, 3),
 'T081-172603-IW2': (4, 3, 3),
 'T081-172603-IW3': (4, 3, 3)}